<a href="https://colab.research.google.com/github/kimsaeyoen/PimaIndians-DecisionTree/blob/main/%ED%8A%B9%EC%A0%95%EC%8B%9C%EA%B0%84_%EC%BA%A1%EC%B2%98_%EB%B0%8F_roi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import cv2

input_directory = '/content/drive/MyDrive/ul_input/12'
output_directory = '/content/drive/MyDrive/ul_output/Mmode_ROI'
roi = (51, 558, 1157, 895)  # (start_x, start_y, end_x, end_y)
capture_time = 18.9  # 캡처할 시간 (초)

def ensure_dir(file_path):
    if not os.path.exists(file_path):
        os.makedirs(file_path)

def capture_frame_and_crop(input_dir, output_dir, roi, capture_time):
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(('.wmv', '.mp4', '.avi', '.mkv', '.mov')):  # 처리할 파일 확장자들
                input_file_path = os.path.join(root, file)
                relative_path = os.path.relpath(root, input_dir)
                output_file_dir = os.path.join(output_dir, relative_path)
                ensure_dir(output_file_dir)
                output_image_path = os.path.join(output_file_dir, os.path.splitext(file)[0] + '_capture.png')

                capture = cv2.VideoCapture(input_file_path)
                if not capture.isOpened():
                    print(f"Error: Cannot open video file {input_file_path}")
                    continue

                fps = capture.get(cv2.CAP_PROP_FPS)
                frame_number = int(capture_time * fps)  # 캡처할 프레임 번호 계산
                capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number)  # 해당 프레임으로 이동

                ret, frame = capture.read()
                if not ret:
                    print(f"Error: Cannot capture frame at {capture_time} seconds from {file}")
                    capture.release()
                    continue

                # Ensure ROI is within frame bounds
                height, width, _ = frame.shape
                if roi[3] > height or roi[2] > width:
                    print(f"Error: ROI {roi} is out of bounds for video {file}")
                    capture.release()
                    continue

                cropped_frame = frame[roi[1]:roi[3], roi[0]:roi[2]]
                cv2.imwrite(output_image_path, cropped_frame)
                print(f"Captured and cropped image saved to {output_image_path}")

                capture.release()

if __name__ == "__main__":
    capture_frame_and_crop(input_directory, output_directory, roi, capture_time)


Captured and cropped image saved to /content/drive/MyDrive/ul_output/Mmode_ROI/01/110114015/20231201140125409/110114015_capture.png
Captured and cropped image saved to /content/drive/MyDrive/ul_output/Mmode_ROI/04/030397765 윤창석/20231204135235148/030397765 윤창석_capture.png
Captured and cropped image saved to /content/drive/MyDrive/ul_output/Mmode_ROI/05/971127395/20231205134519143/971127395_capture.png
Captured and cropped image saved to /content/drive/MyDrive/ul_output/Mmode_ROI/08/040646515 권승묵/20231208135605859/040646515 권승묵_capture.png
Captured and cropped image saved to /content/drive/MyDrive/ul_output/Mmode_ROI/08/220050405 이억영/20231208105845839/220050405 이억영_capture.png
Captured and cropped image saved to /content/drive/MyDrive/ul_output/Mmode_ROI/08/190164265 신금철/20231208135139587/190164265 신금철_capture.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from google.colab import drive
from moviepy.editor import VideoFileClip
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image

In [ ]:
import os
import cv2
from moviepy.editor import VideoFileClip

def crop_image(image, roi):
    """
    주어진 ROI 좌표를 사용하여 이미지를 크롭합니다.

    :param image: 입력 이미지 (OpenCV 형식)
    :param roi: ROI 좌표 (start_y, end_y, start_x, end_x)
    :return: 크롭된 이미지
    """
    start_y, end_y, start_x, end_x = roi
    return image[start_y:end_y, start_x:end_x]

def extract_frame_at_time(video_path, output_dir, capture_time=18, roi=None):
    """
    동영상의 특정 시간(초)에 해당하는 프레임을 추출하여 이미지 파일로 저장합니다.

    :param video_path: 동영상 파일 경로
    :param output_dir: 프레임이 저장될 디렉토리
    :param capture_time: 추출할 시간(초)
    :param roi: 크롭할 ROI 좌표 (start_y, end_y, start_x, end_x)
    """
    print(f"Processing video: {video_path}")

    # 동영상 파일명 추출
    video_filename = os.path.splitext(os.path.basename(video_path))[0]

    # 동영상 로드
    try:
        clip = VideoFileClip(video_path)
    except Exception as e:
        print(f"Error loading video file: {e}")
        return

    duration = clip.duration

roi = (51,558,1157,895)  # 크롭할 ROI 좌표

      try:
        frame = clip.get_frame(capture_time)
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

                if roi is not None:
                frame_bgr = crop_image(frame_bgr, roi)

        frame_filename = os.path.join(output_dir, f"{video_filename}_frame_at_{capture_time:.2f}s.png")
        cv2.imwrite(frame_filename, frame_bgr)
        print(f"Saved frame at {capture_time} seconds: {frame_filename}")
    except Exception as e:
        print(f"Error processing frame at {capture_time} seconds: {e}")

def process_videos_in_folder(folder_path, output_dir, capture_time=18, roi=None):
    """
    폴더 내의 모든 비디오 파일을 처리하여 특정 시간(초)에 해당하는 프레임을 추출합니다.

    :param folder_path: 비디오 파일이 포함된 폴더 경로
    :param output_dir: 프레임이 저장될 디렉토리
    :param capture_time: 추출할 시간(초)
    :param roi: 크롭할 ROI 좌표 (start_y, end_y, start_x, end_x)
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.wmv'):  # 다양한 비디오 형식 추가 가능
                video_path = os.path.join(root, file)
                video_output_dir = os.path.join(output_dir, os.path.splitext(file)[0])
                if not os.path.exists(video_output_dir):
                    os.makedirs(video_output_dir)
                extract_frame_at_time(video_path, video_output_dir, capture_time, roi)

# 사용 예제
input_dir = '/content/MyDrive/MyDrive/ul_input'
output_dir = '/content/MyDrive/MyDrive/ul_output/test_output'
capture_time = 18  # 18초에 해당하는 프레임을 추출

# 폴더 내의 비디오 파일 처리
process_videos_in_folder(input_dir, output_dir, capture_time, roi)


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 51)